In [ ]:
from google.colab import drive
import sys

drive.mount("/content/drive")
sys.path.append("/content/drive/MyDrive/colab_env/lib/python3.11/site-packages")

Mounted at /content/drive


In [ ]:
import great_expectations as ge
import dask.dataframe as dd
import pandas as pd

In [ ]:
CSV_DATA = "/content/drive/MyDrive/laptop_data_2.csv"

In [ ]:
df = pd.read_csv(CSV_DATA)
df.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')

In [ ]:
for col in df.columns:
  print(df[col].value_counts(dropna=False))

Company
MSI       1429821
Apple     1428790
Lenovo    1428689
Acer      1428659
HP        1428487
Asus      1427974
Dell      1427580
Name: count, dtype: int64
TypeName
Ultrabook             2001264
Gaming                2000488
Netbook               2000434
Notebook              1999142
2 in 1 Convertible    1998672
Name: count, dtype: int64
Inches
12.4    159841
15.1    159389
13.1    159379
16.7    159363
16.9    159354
         ...  
14.9    158039
11.8    157872
15.3    157675
17.3     79096
11.0     78806
Name: count, Length: 64, dtype: int64
ScreenResolution
3840x2160    2502743
1366x768     2499912
1920x1080    2498838
3200x1800    2498507
Name: count, dtype: int64
Cpu
AMD Ryzen 5      2000804
Apple M1         2000435
Intel Core i5    1999937
Intel Core i7    1999770
Intel Celeron    1999054
Name: count, dtype: int64
Ram
16GB      2477603
4GB       2476560
8GB       2473675
32GB      2472162
1234TB      50057
NaN         49943
Name: count, dtype: int64
Memory
2TB SSD      20008

### 1. Create Expectation Suite (from small sample)

In [ ]:
print(ge.__version__)

1.5.2


In [ ]:
import great_expectations as gx

# Get the Ephemeral Data Context
context = gx.get_context()
assert type(context).__name__ == "EphemeralDataContext"

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpyqtiq7mi' for ephemeral docs site


In [ ]:
# Add a Pandas Data Source
#data_source = context.data_sources.add_pandas(name="laptop_price_prediction")
# Add a Data Asset to the Data Source
data_asset = data_source.add_dataframe_asset(name="laptop_price_prediction_asset")

### 2. Adding a batch definition

In [ ]:
# Define the Batch Definition name
batch_definition_name = "laptop_price_prediction_batch"
# Add the Batch Definition
batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)
assert batch_definition.name == batch_definition_name

### 3. Retrieving a batch

In [ ]:
# Define the Batch Parameters
batch_parameters = {"dataframe": df}
# Retrieve the Batch
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

### 4. Creating a suite and defining expectations

In [ ]:
expectation_suite_name = "laptop_price_prediction_suite"
suite = gx.ExpectationSuite(name=expectation_suite_name)

# Define expectations as a list of dicts
price_expectations = [
    gx.expectations.ExpectColumnValuesToNotBeNull(column="Price"),
    gx.expectations.ExpectColumnValuesToBeOfType(column="Price", type_="float64"),
    gx.expectations.ExpectColumnValuesToBeBetween(column="Price", min_value=0, strict_min=False)
]

for expectation in price_expectations:
    suite.add_expectation(expectation)

In [ ]:
# Validate the Data Against the Suite

# Evaluate the Results
#print(validation_results)

try:
    validation_results = batch.validate(suite)
    print(validation_results)
except Exception as e:
    print("❌ Validation failed with an exception:")
    print(f"🔎 {type(e).__name__}: {str(e)}")

Calculating Metrics:   0%|          | 0/23 [00:00<?, ?it/s]

{
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_of_type",
        "kwargs": {
          "column": "Price",
          "type_": "number",
          "batch_id": "laptop_price_prediction-laptop_price_prediction_asset"
        },
        "meta": {},
        "id": "46764145-ea92-4c9c-a325-515deb574d13"
      },
      "result": {},
      "meta": {},
      "exception_info": {
        "MetricConfigurationID(metric_name='column_values.of_type.condition', metric_domain_kwargs_id='443c8ea7f0017af12a4ee6449bdef1a2', metric_value_kwargs_id='type_=number')": {
          "exception_traceback": "Traceback (most recent call last):\n  File \"/usr/local/lib/python3.11/dist-packages/great_expectations/execution_engine/execution_engine.py\", line 534, in _process_direct_and_bundled_metric_computation_configurations\n    metric_computation_configuration.metric_fn(  # type: ignore[misc] # F not callable\n  File \"/usr

In [ ]:
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Price"] = df["Price"].apply(lambda x: 0 if pd.isna(x) or x < 0 else x)

df["Price"].value_counts()

,count
Price,
0.00,100000
1447.02,68
2735.17,68
2614.31,65
2155.90,65
...,...
1210.79,14
2351.88,14
1058.73,14


In [ ]:
validation_results = batch.validate(suite)
print(validation_results)

Calculating Metrics:   0%|          | 0/14 [00:00<?, ?it/s]

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "laptop_price_prediction-laptop_price_prediction_asset",
          "column": "Price"
        },
        "meta": {},
        "id": "c19630e7-fb92-4d26-a5d8-a93a2061a96b"
      },
      "result": {
        "element_count": 10000000,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_of_type",
        "kwargs": {
          "batch_id": "laptop_price_prediction-laptop_price_prediction_asset",
 

In [ ]:
if validation_results["success"] == False:
    for r in validation_results["results"]:
        #print(r.expectation_config)
        if r["success"] == False:
            print(f"❌ Failed: {r.expectation_config.type}")
            print(f"  Column: {r.expectation_config.kwargs.get('column')}")
            print(f"  Result details: {r['result']}")
            print(f"  Exception: {r['exception_info']}")